# Lab 2.2 Classification

> **From the slide to the code** — slide *Learning from vectors*
> - "Supervised: with labels → predict them" (Decision Tree, SVM, Random Forest, Naive Bayes) → all from scikit-learn: `tree`, `svm`, `ensemble`, `naive_bayes`
> - every classifier has the same two steps: `clf.fit(X_train_vec, y_train)` (learn from labelled documents) and `clf.predict(X_test_vec)` (predict new ones)

Classification is a **supervised** learning task: from documents with a label (spam / ham, positive / negative) the model learns to predict the label of new documents. In Lab 2.1 the labels were used only to *evaluate* the clusters; here they are used to **train** the model.

We load the SMS of Lab 2.1 and compare three classifiers (Decision Tree, SVC and Random Forest) with two vectorizers (Count and TF-IDF).

In [1]:
!pip install gdown

In [2]:
!gdown 'https://drive.google.com/uc?id=1SwQEuHyO2A9hfSK2rdRH0J-YQ426U_20'

Downloading...
From: https://drive.google.com/uc?id=1SwQEuHyO2A9hfSK2rdRH0J-YQ426U_20
To: /content/spam.csv
100% 479k/479k [00:00<00:00, 7.47MB/s]


In [3]:
import pandas as pd
import numpy as np

spam_data = pd.read_csv('spam.csv')

spam_data['target'] = np.where(spam_data['target'] == 'spam', 1, 0)
spam_data.head(10)

,text,target
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0
5,FreeMsg Hey there darling it's been 3 week's n...,1
6,Even my brother is not like to speak with me. ...,0
7,As per your request 'Melle Melle (Oru Minnamin...,0
8,WINNER!! As a valued network customer you have...,1
9,Had your mobile 11 months or more? U R entitle...,1


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, recall_score

# 75% of the SMS to train, 25% to test (random_state: the same split every run)
X, y = spam_data['text'], spam_data['target']
X_train, X_test, y_train, y_test = train_test_split(spam_data['text'],
                                                    spam_data['target'],
                                                    random_state=0)

In [5]:
spam_percent = len(spam_data[spam_data['target'] == 1]) / len(spam_data) * 100
print("Spam percentage %s " %spam_percent)

Spam percentage 13.406317300789663 


In [6]:
# Count vectors: vocabulary learned on the training SMS only
vectorizer = CountVectorizer()
X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [7]:
X_train.shape

(4179,)

In [8]:
X_train_transformed.shape

(4179, 7354)

## Decision Trees, SVC and Random Forests

> **From the slide to the code** — slide *Classic classifiers / Classifiers in Python*
> - Decision Tree: "splits on one feature at a time, easy to interpret" → `DecisionTreeClassifier(random_state=0)`
> - SVC: "max-margin hyperplane; default kernel `rbf` (non-linear); for text, `kernel=\"linear\"` is usual" → `SVC()` / `SVC(kernel='linear')`
> - Random Forest: "many trees, majority vote" (100 trees on bootstrap samples) → `RandomForestClassifier(random_state=0)`


### DecisionTreeClassifier
A Decision Tree is a model that uses a tree-like structure to make decisions based on input features. It starts with a root node and splits data into subsets based on feature values, using a criterion like Gini impurity or Information Gain.
Each node represents a feature, and branches represent decision rules.
The process continues until a stopping condition is met (e.g., maximum depth or pure leaf nodes).

### SVC (Support Vector Classifier)
SVC is part of the Support Vector Machine (SVM) family. It aims to find the best hyperplane that separates classes in the feature space. It maps data points to a high-dimensional space (using kernels if necessary).
Finds the hyperplane that maximizes the margin (distance) between different classes.
Supports both linear and non-linear classification.

### RandomForestClassifier
A Random Forest is an ensemble learning method based on a collection of decision trees.
It creates multiple decision trees during training and combines their outputs for prediction.
Uses techniques like bagging (bootstrap aggregation) and feature randomness to improve generalization.

### When to Use Each
- **DecisionTreeClassifier:** When interpretability is crucial, or for small datasets with simple relationships.
- **SVC:** For datasets with clear class boundaries or when kernel methods are needed for non-linear separability.
- **RandomForestClassifier:** When performance is the priority, especially on larger datasets with complex patterns.

## Precision, recall and F1

> **From the slide to the code** — slide *Precision & recall*
> - TP = spam → spam, FP = ham → spam, FN = spam → ham, TN = ham → ham
> - P = TP / (TP + FP), R = TP / (TP + FN), F1 = 2·P·R / (P + R) → `precision_score`, `recall_score`, `f1_score` (for the class 1 = spam); all at once: `classification_report(y_test, y_pred)`

We compare each prediction with the true label, for the **spam** class:
- **Precision**: of the SMS flagged as spam, how many are really spam?
- **Recall**: of the real spam, how much did we find?
- **F1**: their harmonic mean, high only if both are high.

**Why not accuracy?** 86.6% of the SMS are ham: a model that always answers "ham" has 86.6% accuracy, but it finds no spam at all (recall = F1 = 0). With more than two classes, F1 is averaged over the classes: **macro** (every class counts the same) or **micro** (every document counts the same).

In [9]:
def evaluate_models(X_tr, y_tr, X_te, y_te):
    """Train the three classifiers and print the same metrics for each one."""
    models = {
        "Decision Tree": DecisionTreeClassifier(random_state=0),
        "SVM": SVC(),
        "RF": RandomForestClassifier(random_state=0),
    }
    for name, clf in models.items():
        y_pred = clf.fit(X_tr, y_tr).predict(X_te)   # learn on the training SMS, predict the test SMS
        print(name)
        print(f"   Accuracy: {accuracy_score(y_te, y_pred):.4f}")
        print(f"   Recall:   {recall_score(y_te, y_pred):.4f}")
        print(f"   F1 score: {f1_score(y_te, y_pred):.4f}")

# Given: the three models on the Count vectors
evaluate_models(X_train_transformed, y_train, X_test_transformed, y_test)

Decision Tree
   Accuracy: 0.9670
   Recall:   0.8376
   F1 score: 0.8777


SVM
   Accuracy: 0.9785
   Recall:   0.8477
   F1 score: 0.9176


RF
   Accuracy: 0.9727
   Recall:   0.8071
   F1 score: 0.8933


# Exercise 2.1: Count or TF-IDF? Which kernel?

The cell above gives the results of the three models with **Count** vectors (`evaluate_models`).

1. Vectorize the SMS again with `TfidfVectorizer` (fit on `X_train`, transform `X_train` and `X_test`) and run `evaluate_models` on the new vectors.
2. The slides say that for text a **linear** kernel is usual: train `SVC(kernel='linear')` on the Count and on the TF-IDF vectors and print accuracy, precision, recall and F1.
3. Which model and which vectors would you choose? Look at the recall and the F1, not only at the accuracy: why?

*Hint: `from sklearn.metrics import precision_score`.*

In [10]:
# Your code here

## K-Fold Cross-Validation

> **From the slide to the code** — slide *Cross-validation*
> - "One train/test split can be lucky or unlucky. K-Fold: split the data into k folds; train on k−1, test on the last one; repeat k times and average"
> - "Stratified: each fold keeps the class proportions" → `skf = StratifiedKFold(n_splits=5)`, `for train_idx, test_idx in skf.split(X, y): ...`; report **mean ± std**

In K-Fold Cross-Validation the dataset is divided into k equally sized folds. The model is trained and tested k times: each fold is the test set once, while the other k−1 folds are the training set. Every document is used both for training and for testing, and the k scores are averaged: a more stable estimate than a single split.

Careful: tune the model with cross-validation **on the training set**, and keep the test set for the very end.

## Stratified K-Fold

Stratified K-Fold is a popular technique in machine learning for evaluating model performance while ensuring a balanced representation of target classes across the folds. This method is particularly useful when dealing with imbalanced datasets, where one class significantly outnumbers others.

In standard K-Fold cross-validation, the folds are cut without looking at the labels. However, standard K-Fold may result in an uneven distribution of target class labels across the folds, leading to biased or unreliable model evaluation, especially in classification tasks.

Stratified K-Fold addresses this issue by ensuring that each fold maintains the same proportion of target class labels as the original dataset. This stratification helps provide a more robust and representative evaluation of the model's performance, especially in scenarios where class imbalance could skew results.

By combining the benefits of stratification with cross-validation, Stratified K-Fold offers a reliable framework for assessing and optimizing machine learning models, particularly for classification tasks.

# Exercise 2.2: cross-validation of the three models

1. Use `StratifiedKFold(n_splits=5)` on all the SMS (`X`, `y`). In each fold, fit a `TfidfVectorizer` on the training part only, then train and test the three models (Decision Tree, SVC, Random Forest) and keep the **F1** of each model.
2. Print the F1 of each model as **mean ± std** over the 5 folds. Which model is the best? Which is the most stable?
3. Check that every test fold keeps the share of spam of the whole dataset (13.4%).

*Hint: `X.iloc[train_index]` selects the SMS of the training part of a fold; `np.mean(list)`, `np.std(list)`.*

In [14]:
# Your code here

## Naive Bayes: Amazon reviews

> **From the slide to the code** — slide *Naive Bayes*
> - "Bayes: P(class | words) ∝ P(class) · ∏ P(word | class). Naive: words are independent given the class" → `MultinomialNB().fit(X_train, y_train)`
> - "keep `not`, `but`: `not good`, `good but`" → `STOPWORDS.remove('not')`; `TfidfVectorizer(min_df=5, ngram_range=(1, 2))`; "look at each class, not only at accuracy" → `classification_report`

New data: about 50,000 Amazon reviews of video games. A review is **positive** if it has more than 3 stars. The next cells are given: download, cleaning, and the label `positive`.

In [18]:
# download the reviews
# Amazon Reviews 2018 (McAuley lab, UCSD), category Video Games, 5-core: 497,577 reviews
!wget -q --show-progress https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/categoryFilesSmall/Video_Games_5.json.gz
# download preprocessing_functions.py from the course repository, only if it is not already here
![ -f preprocessing_functions.py ] || wget -q "https://raw.githubusercontent.com/steestee25/tms_lab/main/Lab2/preprocessing_functions.py" -O preprocessing_functions.py

Video_Games_5.json.     [<=>                 ]       0  --.-KB/s               

Video_Games_5.json.     [ <=>                ] 102.54K  --.-KB/s    in 0.06s   


In [19]:
!ls

In [20]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords  # preprocessing_functions downloads the stopwords if needed

from preprocessing_functions import remove_punctuation, remove_stopwords, remove_urls

In [21]:
data = "Video_Games_5.json.gz"
df = pd.read_json(data, lines = True, compression = "gzip")
df.head()

,overall,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,vote,style,image
0,5,True,"10 17, 2015",A1HP7NVNPFMA4N,0700026657,Ambrosia075,"This game is a bit hard to get the hang of, bu...",but when you do it's great.,1445040000,NaN,NaN,NaN
1,4,False,"07 27, 2015",A1JGAP0185YJI6,0700026657,travis,I played it a while but it was alright. The st...,"But in spite of that it was fun, I liked it",1437955200,NaN,NaN,NaN
2,3,True,"02 23, 2015",A1YJWEXHQBWK2B,0700026657,Vincent G. Mezera,ok game.,Three Stars,1424649600,NaN,NaN,NaN
3,2,True,"02 20, 2015",A2204E1TH211HT,0700026657,Grandma KR,"found the game a bit too complicated, not what...",Two Stars,1424390400,NaN,NaN,NaN
4,5,True,"12 25, 2014",A2RF5B5H74JLPE,0700026657,jon,"great game, I love it and have played it since...",love this game,1419465600,NaN,NaN,NaN


In [22]:
# Initial size
print(df.shape)

df = df[df.reviewText.notnull()].reset_index(drop=True)
print(df.shape)

(497577, 12)


(497419, 12)


In [23]:
df = df[df.verified].reset_index(drop=True)
df.shape

(332504, 12)

In [24]:
df = df.sample(50_000, random_state=42).reset_index(drop=True)  # random_state: same sample every run

In [25]:
# 'not' and 'but' can be relevant to classify reviews as positive / negative
# E.g., bigrams "not good", "good but"
STOPWORDS = stopwords.words('english')
STOPWORDS.remove('not')
STOPWORDS.remove('but')

# preprocess dataset

def preprocess(text):

    text = text.lower()
    text = re.sub(r'^<div id="video.*>&nbsp;', '', text) # Video-review part
    text = remove_urls(text)
    text = remove_punctuation(text)
    text = remove_stopwords(text, STOPWORDS)
    # Remove words that are digits only (raw string r'...': otherwise \b is a backspace)
    text = re.sub(r'\b\d+\b', '', text)
    text = " ".join(text.split())  # remove the extra spaces left behind

    return text

In [26]:
%%time
df["preprocessed"] = df.reviewText.apply(preprocess)
# Will take a few seconds

CPU times: user 4.51 s, sys: 4.8 ms, total: 4.52 s
Wall time: 4.59 s


In [27]:
df = df[~df.preprocessed.str.contains(r"^\s*$")]

In [28]:
df["positive"] = df.overall > 3

print(df.groupby("positive").size()/(df.shape[0]))

positive
False    0.165734
True     0.834266
dtype: float64


## MultinomialNB

MultinomialNB is a type of Naive Bayes classifier specifically designed for discrete data, such as word counts in text classification. It is part of the Naive Bayes family of algorithms, which are based on the principle of Bayes' Theorem with the assumption that features are conditionally independent given the class label.

The next two cells are given: train/test split (67% / 33%) and TF-IDF vectors of words **and bigrams** (`ngram_range=(1, 2)`), keeping only the terms that appear in at least 5 reviews (`min_df=5`).

In [29]:
from sklearn.metrics import classification_report # good way of presenting results
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.preprocessed.values,
                                                    df.positive.values,
                                                    test_size=.33,
                                                    random_state=42
                                                    )

In [30]:
vectorizer = TfidfVectorizer(min_df=5, ngram_range=(1, 2))
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Exercise 2.3: Naive Bayes and the accuracy trap

1. Train a `MultinomialNB` on `X_train_vectorized`, `y_train`, predict `X_test_vectorized` and print the `classification_report` (with `digits=3`).
2. Look at the accuracy and at the **recall of the negative class** (`False`). What does the model do? Why?
3. Which words best **separate** the classes? Use `clf.feature_log_prob_` (one row per class: log P(word | class)) and `vectorizer.get_feature_names_out()`.
4. Try `MultinomialNB(fit_prior=False)` and `ComplementNB()`: what happens to the recall of the negative class?

*Hint: the most frequent words of a class are not the ones that separate the classes: look at the **difference** `feature_log_prob_[1] - feature_log_prob_[0]`.*

In [31]:
# Your code here